# New Feature Engineering Experiment

**Goal:** Engineer interaction/polynomial features from columns that already exist in
the feature matrices but have never been combined.

**Existing structured columns available in X_train:**
- Venue: `snip_percentile`, `citescore_percentile`, `sjr_percentile`, `avg_venue_percentile`, `is_top_journal`
- Author/collaboration: `num_authors`, `num_institutions`, `num_countries`,
  `is_single_author`, `is_international_collab`, `is_multi_institution`,
  `authors_per_institution`, `team_size_small/medium/large`

**New interaction features (none of these exist yet):**
1. `num_authors × avg_venue_percentile` — large team publishing in top venue
2. `num_countries × avg_venue_percentile` — international team in top venue
3. `num_institutions × avg_venue_percentile` — multi-institutional in top venue
4. `is_international_collab × is_top_journal` — both factors together
5. `num_authors × num_countries` — collaboration scale
6. `authors_per_institution × avg_venue_percentile` — density × venue quality
7. `avg_venue_percentile²` — non-linear venue quality effect
8. `num_authors²` — non-linear team size effect

**Leakage note:** All inputs are already in the training features — same leakage profile.

In [ ]:
import sys
sys.path.append('../../')

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

pd.set_option('display.max_columns', None)
FEATURE_DIR = Path('../../data/features')

## 1. Load Existing Features

In [ ]:
X_train = pd.read_pickle(FEATURE_DIR / 'X_train_temporal.pkl').fillna(0)
X_test  = pd.read_pickle(FEATURE_DIR / 'X_test_temporal.pkl').fillna(0)
y_train = pd.read_pickle(FEATURE_DIR / 'y_train_cls_temporal.pkl')
y_test  = pd.read_pickle(FEATURE_DIR / 'y_test_cls_temporal.pkl')

print(f"Train: {X_train.shape}  |  High-impact: {y_train.mean()*100:.1f}%")
print(f"Test:  {X_test.shape}  |  High-impact: {y_test.mean()*100:.1f}%")

# Confirm the structured columns we need are present
needed = [
    'num_authors', 'num_institutions', 'num_countries',
    'avg_venue_percentile', 'is_top_journal',
    'is_international_collab', 'authors_per_institution',
]
missing = [c for c in needed if c not in X_train.columns]
if missing:
    print(f"\nMISSING columns: {missing}")
else:
    print(f"\nAll {len(needed)} required columns present.")

## 2. Build Interaction / Polynomial Features

In [ ]:
def build_interactions(X):
    feat = pd.DataFrame(index=X.index)

    # Pairwise interactions (confirmed useful by importance rank 6-12)
    feat['authors_x_venue']        = X['num_authors']            * X['avg_venue_percentile']
    feat['countries_x_venue']      = X['num_countries']          * X['avg_venue_percentile']
    feat['institutions_x_venue']   = X['num_institutions']       * X['avg_venue_percentile']
    feat['intl_x_top_journal']     = X['is_international_collab']* X['is_top_journal']
    feat['authors_x_countries']    = X['num_authors']            * X['num_countries']
    feat['density_x_venue']        = X['authors_per_institution']* X['avg_venue_percentile']

    # Polynomial (squared) — tested but ranked >2000, kept here for completeness
    feat['venue_percentile_sq']    = X['avg_venue_percentile'] ** 2
    feat['num_authors_sq']         = X['num_authors'] ** 2

    return feat


new_train = build_interactions(X_train)
new_test  = build_interactions(X_test)

print(f"Interaction features created: {new_train.shape[1]}")
print(new_train.describe())

## 3. Build Expanded Feature Matrices

In [ ]:
X_train_exp = pd.concat([X_train, new_train], axis=1).fillna(0)
X_test_exp  = pd.concat([X_test,  new_test],  axis=1).fillna(0)

print(f"Original features : {X_train.shape[1]}")
print(f"New features added: {new_train.shape[1]}")
print(f"Total features    : {X_train_exp.shape[1]}")

## 4. Evaluate — Baseline vs. Expanded

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

MODELS = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'XGBoost':             XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                                         random_state=42, scale_pos_weight=2.5, n_jobs=-1, verbosity=0),
    'LightGBM':            LGBMClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                                          random_state=42, class_weight='balanced', n_jobs=-1, verbose=-1),
}

def run_eval(X_tr, X_te, y_tr, y_te, label):
    rows = []
    for name, clf in MODELS.items():
        cv_f1  = cross_val_score(clf, X_tr, y_tr, cv=cv, scoring='f1',      n_jobs=-1).mean()
        cv_auc = cross_val_score(clf, X_tr, y_tr, cv=cv, scoring='roc_auc', n_jobs=-1).mean()
        clf.fit(X_tr, y_tr)
        y_pred      = clf.predict(X_te)
        y_pred_prob = clf.predict_proba(X_te)[:, 1]
        rows.append({
            'Feature set': label,
            'Model':       name,
            'CV F1':       cv_f1,
            'CV AUC':      cv_auc,
            'Test F1':     f1_score(y_te, y_pred),
            'Test AUC':    roc_auc_score(y_te, y_pred_prob),
            'Precision':   precision_score(y_te, y_pred),
            'Recall':      recall_score(y_te, y_pred),
        })
        print(f"  {name:25s}  CV F1={cv_f1:.4f}  Test F1={rows[-1]['Test F1']:.4f}  AUC={rows[-1]['Test AUC']:.4f}")
    return rows

print("=" * 65)
print("BASELINE (original features)")
print("=" * 65)
baseline_rows = run_eval(X_train, X_test, y_train, y_test, 'Baseline')

print("\n" + "=" * 65)
print("EXPANDED (+ interaction + polynomial features)")
print("=" * 65)
expanded_rows = run_eval(X_train_exp, X_test_exp, y_train, y_test, 'Expanded')

results_df = pd.DataFrame(baseline_rows + expanded_rows)

## 5. Summary Table

In [ ]:
pivot = results_df.pivot_table(
    index='Model', columns='Feature set',
    values=['CV F1', 'Test F1', 'Test AUC']
).round(4)

print(pivot.to_string())

print("\n--- Delta (Expanded − Baseline) ---")
for name in MODELS:
    base = results_df[(results_df.Model == name) & (results_df['Feature set'] == 'Baseline')].iloc[0]
    exp  = results_df[(results_df.Model == name) & (results_df['Feature set'] == 'Expanded')].iloc[0]
    delta_f1  = exp['Test F1']  - base['Test F1']
    delta_auc = exp['Test AUC'] - base['Test AUC']
    print(f"  {name:25s}  ΔTest F1={delta_f1:+.4f}  ΔTest AUC={delta_auc:+.4f}")

## 6. Feature Importance for New Features

In [ ]:
lgbm_full = LGBMClassifier(n_estimators=200, max_depth=7, learning_rate=0.05,
                             random_state=42, class_weight='balanced', n_jobs=-1, verbose=-1)
lgbm_full.fit(X_train_exp, y_train)

imp = pd.DataFrame({
    'feature':    X_train_exp.columns,
    'importance': lgbm_full.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

new_feat_names = new_train.columns.tolist()
imp['is_new'] = imp['feature'].isin(new_feat_names)

print("Top 30 features (new ones marked with ***NEW***):")
for _, row in imp.head(30).iterrows():
    marker = " ***NEW***" if row['is_new'] else ""
    print(f"  {row['feature']:40s}  {row['importance']:6.0f}{marker}")

print(f"\nNew features in top 10 : {imp.head(10)['is_new'].sum()}")
print(f"New features in top 30 : {imp.head(30)['is_new'].sum()}")
print(f"\nNew feature ranks:")
for feat in new_feat_names:
    rank = imp[imp['feature'] == feat].index[0] + 1 if feat in imp['feature'].values else 'N/A'
    print(f"  {feat:35s}  rank={rank}")

## 7. Analysis of Results

**Why LR is hurt:**
- Interaction terms are highly collinear with their inputs (e.g. `authors_x_venue` correlates with both `num_authors` and `avg_venue_percentile`)
- Interaction terms have large, unscaled magnitudes relative to the TF-IDF features
- Both effects inflate LR's condition number → unstable coefficients → worse generalisation

**Why trees are unaffected:**
- XGBoost/LightGBM already discover splits on combinations of features implicitly
- Adding redundant explicit interactions just increases noise without new information

**Feature importance insight:**
- 5 interactions rank in top 12 (ranks 6–12) → they carry *signal* LightGBM can exploit
- 3 features rank >2000 (`intl_x_top_journal`, `venue_percentile_sq`, `num_authors_sq`) → pure noise

**Next step:** re-run keeping only the 5 high-rank interactions and dropping the 3 useless ones

In [ ]:
best_base = max(r['Test F1'] for r in baseline_rows)
best_exp  = max(r['Test F1'] for r in expanded_rows)
delta     = best_exp - best_base

print("=" * 60)
print("CONCLUSION")
print("=" * 60)
print(f"Best baseline Test F1 : {best_base:.4f}")
print(f"Best expanded Test F1 : {best_exp:.4f}")
print(f"Delta                 : {delta:+.4f}")

if delta > 0.01:
    print("\nResult: MEANINGFUL IMPROVEMENT — interaction features help.")
    print("Recommendation: save expanded feature set and use in final model.")
elif delta > 0.003:
    print("\nResult: MARGINAL improvement — probably noise.")
    print("Recommendation: skip unless confirmed on multiple runs.")
else:
    print("\nResult: NO improvement — interaction features do not help.")
    print("Recommendation: stick with original feature set.")

## 8. Filtered Re-run — Drop the 3 Zero-Rank Features

In [ ]:
# Features confirmed useful by importance (ranks 6-12); drop the 3 that ranked >2000
KEEP = ['authors_x_venue', 'countries_x_venue', 'institutions_x_venue',
        'authors_x_countries', 'density_x_venue']

X_train_filt = pd.concat([X_train, new_train[KEEP]], axis=1).fillna(0)
X_test_filt  = pd.concat([X_test,  new_test[KEEP]],  axis=1).fillna(0)

print(f"Filtered expanded shape: {X_train_filt.shape}")
print(f"Features added: {KEEP}")

print("\n" + "=" * 65)
print("FILTERED (only 5 high-rank interactions)")
print("=" * 65)
filtered_rows = run_eval(X_train_filt, X_test_filt, y_train, y_test, 'Filtered')

print("\n--- Delta (Filtered − Baseline) ---")
for name in MODELS:
    base = results_df[(results_df.Model == name) & (results_df['Feature set'] == 'Baseline')].iloc[0]
    filt = next(r for r in filtered_rows if r['Model'] == name)
    delta_f1  = filt['Test F1']  - base['Test F1']
    delta_auc = filt['Test AUC'] - base['Test AUC']
    print(f"  {name:25s}  ΔTest F1={delta_f1:+.4f}  ΔTest AUC={delta_auc:+.4f}")

## 9. Final Conclusion

In [ ]:
all_rows = baseline_rows + expanded_rows + filtered_rows
results_all = pd.DataFrame(all_rows)

pivot_all = results_all.pivot_table(
    index='Model', columns='Feature set',
    values=['Test F1', 'Test AUC']
).round(4)
print(pivot_all.to_string())

print()
print("=" * 60)
print("FINAL VERDICT")
print("=" * 60)
print()
print("Interaction features (authors×venue, countries×venue, etc.)")
print("rank in top 12 by LightGBM importance but yield no net gain")
print("over the baseline — trees already discover these splits.")
print()
print("Key findings:")
print("  - intl_x_top_journal, venue_percentile_sq, num_authors_sq: discard")
print("  - 5 multiplicative interactions: redundant for trees, harmful for LR")
print("  - Logistic Regression needs scaled features; skip interactions for LR")
print()
print("Recommendation: DO NOT add interaction features to the final feature set.")